# Data Preparation

Questo notebook implementa la pipeline di preparazione dei dati utilizzati
per l'addestramento dei modelli di predizione delle partite di League of Legends.

La pipeline comprende:

1. caricamento del dataset originale;
2. rimozione delle feature collineari e ridondanti;
3. separazione della variabile target `blueWins`;
4. suddivisione in Training Set e Test Set;
5. standardizzazione delle feature mediante `StandardScaler`;
6. esportazione dei dataset preprocessati.

Per prevenire fenomeni di data leakage, lo scaler viene addestrato
esclusivamente sul Training Set.


In [35]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [36]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import prep_functions

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\mela\Desktop\predictLoL\predictLoL


## 1. Caricamento del dataset

In [37]:
DATA_PATH = PROJECT_ROOT / "data" / "high_diamond_ranked_10min.csv"

assert DATA_PATH.exists(), f"Dataset non trovato: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

print(f"Shape dataset originale: {df.shape}")

df.head()

Shape dataset originale: (9879, 40)


,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,4519157822,0,28,2,1,9,6,11,0,0,...,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,4523371949,0,12,1,0,5,5,5,0,0,...,1,17620,6.8,17438,240,52,2908,1173,24.0,1762.0
2,4521474530,0,15,0,0,7,11,4,1,1,...,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
3,4524384067,0,43,1,0,4,5,5,1,0,...,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
4,4436033771,0,75,4,0,6,6,6,0,0,...,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4


## 2. Applicazione delle funzioni di preprocessing

Vengono applicate in sequenza le funzioni definite nel modulo
`prep_functions.py` per rimuovere feature collineari, ridondanti e aggregate.

In [38]:
df_clean = prep_functions.rimuovi_collinearita(df)

df_clean = prep_functions.rimuovi_ridondanze_speculari(df_clean)

df_clean = prep_functions.rimuovi_aggregati(df_clean)

print(f"Shape dataset originale: {df.shape}")
print(f"Shape dataset preprocessato: {df_clean.shape}")

Shape dataset originale: (9879, 40)
Shape dataset preprocessato: (9879, 29)


### Rimozione dell'identificatore della partita

La variabile `gameId` rappresenta esclusivamente un identificatore univoco della partita e non contiene informazioni relative allo stato del gioco.

Per evitare che venga utilizzata impropriamente dai modelli come feature predittiva, viene eliminata prima della separazione tra feature e variabile target.

In [39]:
df_clean = df_clean.drop(
    columns=["gameId"],
    errors="ignore"
)

print("gameId presente:", "gameId" in df_clean.columns)
print("Shape dopo la rimozione di gameId:", df_clean.shape)

gameId presente: False
Shape dopo la rimozione di gameId: (9879, 28)


In [40]:
removed_columns = sorted(set(df.columns) - set(df_clean.columns))

print("Feature rimosse:")

for column in removed_columns:
    print(f"- {column}")

Feature rimosse:
- blueAssists
- blueCSPerMin
- blueEliteMonsters
- gameId
- redAssists
- redCSPerMin
- redDeaths
- redEliteMonsters
- redExperienceDiff
- redFirstBlood
- redGoldDiff
- redKills


## 3. Separazione tra feature e target

La variabile `blueWins` rappresenta il target da predire.
Le restanti colonne costituiscono le feature esplicative.

In [41]:
TARGET = "blueWins"

X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

Shape X: (9879, 27)
Shape y: (9879,)


## 4. Train/Test Split

Il dataset viene suddiviso in:
- 80% Training Set;
- 20% Test Set.

Il parametro `random_state=42` garantisce la riproducibilità dello split.
Il Test Set viene mantenuto separato per la valutazione finale del modello.

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (7903, 27)
X_test:  (1976, 27)
y_train: (7903,)
y_test:  (1976,)


## 5. Standardizzazione senza Data Leakage

Lo `StandardScaler` viene addestrato esclusivamente sul Training Set.

Le statistiche calcolate sul Training Set vengono poi utilizzate per
trasformare sia il Training Set sia il Test Set, evitando che informazioni
provenienti dal Test Set influenzino la fase di preprocessing.

In [43]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling completato.")

Scaling completato.


In [44]:
X_train_scaled.describe().T[["mean", "std"]].head(10)

,mean,std
blueWardsPlaced,-2.202745e-17,1.000063
blueWardsDestroyed,-3.506411e-17,1.000063
blueFirstBlood,-2.427515e-17,1.000063
blueKills,7.732086e-17,1.000063
blueDeaths,1.081143e-16,1.000063
blueDragons,1.438528e-17,1.000063
blueHeralds,7.911902e-17,1.000063
blueTowersDestroyed,-1.798160e-17,1.000063
blueTotalGold,-9.080706e-17,1.000063
blueAvgLevel,-2.708927e-15,1.000063


## 6. Esportazione dei dataset

I dataset preprocessati vengono salvati nella cartella `data/`
per essere utilizzati nelle successive fasi di modellazione.

In [45]:
OUTPUT_DIR = PROJECT_ROOT / "data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset preprocessati NON standardizzati
X_train.to_csv(
    OUTPUT_DIR / "X_train.csv",
    index=False
)

X_test.to_csv(
    OUTPUT_DIR / "X_test.csv",
    index=False
)

# Dataset standardizzati
X_train_scaled.to_csv(
    OUTPUT_DIR / "X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    OUTPUT_DIR / "X_test_scaled.csv",
    index=False
)

# Target
y_train.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "y_train.csv",
    index=False
)

y_test.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "y_test.csv",
    index=False
)